In [1]:
# ============================================================
# R3_ROMA_statistical_significance_block_bootstrap.ipynb
# ROMA-TWETF Clean Rebuild: Statistical Inference
#
# Corrected version:
# - Robustly loads AURORA Notebook 10 returns even if date is stored as index.
# - Tests ROMA vs benchmarks.
# - Tests ROMA vs AURORA10 policies when AURORA returns are available.
# - Uses paired circular block bootstrap.
#
# Educational/research use only.
# Not personalized financial advice.
# ============================================================

from __future__ import annotations

import json
import math
import hashlib
import warnings
from pathlib import Path
from datetime import datetime, timezone

warnings.filterwarnings("ignore")

# ============================================================
# 0. Colab setup
# ============================================================

try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception:
    print("Google Drive mount skipped or failed.")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    import seaborn as sns
    HAS_SEABORN = True
except Exception:
    HAS_SEABORN = False

try:
    from scipy import stats
    HAS_SCIPY = True
except Exception:
    HAS_SCIPY = False
    print("scipy not available. HAC p-values will use fallback approximations.")

# ============================================================
# 1. Paths and run configuration
# ============================================================

PROJECT_CODE = "ROMA_TWETF"
AURORA_CODE = "AURORA_TWETF"

PUBLICATION_ROOT = Path("/content/drive/MyDrive/AURORA_TWETF")

# Completed ROMA R2 run
ROMA_R2_RUN_ID = "20260625_025314"

ROMA_R2_ROOT = (
    PUBLICATION_ROOT
    / "outputs"
    / PROJECT_CODE
    / "aligned_allocation_backtest"
    / f"run_{ROMA_R2_RUN_ID}"
)

ROMA_R3_INPUT_INDEX_PATH = ROMA_R2_ROOT / "NOTEBOOK_R3_ROMA_INPUT_INDEX.csv"

# AURORA Notebook 10 outputs
AURORA_NOTEBOOK10_RUN_ID = "20260624_100748"
AURORA_NOTEBOOK11_RUN_ID = "20260624_124834"

AURORA_NOTEBOOK10_ROOT = (
    PUBLICATION_ROOT
    / "outputs"
    / AURORA_CODE
    / "uncertainty_aware_mean_variance_allocation"
    / f"run_{AURORA_NOTEBOOK10_RUN_ID}"
)

AURORA_RETURNS_CANDIDATES = [
    AURORA_NOTEBOOK10_ROOT / "returns" / "comparison_returns_uamv_vs_benchmarks.parquet",
    AURORA_NOTEBOOK10_ROOT / "returns" / "comparison_returns_uamv_vs_benchmarks.csv",
    AURORA_NOTEBOOK10_ROOT / "returns" / "uamv_comparison_returns.parquet",
    AURORA_NOTEBOOK10_ROOT / "returns" / "all_comparison_returns.parquet",
]

OUTPUT_ROOT = PUBLICATION_ROOT / "outputs" / PROJECT_CODE
TABLE_DIR = OUTPUT_ROOT / "tables"
REPORT_DIR = OUTPUT_ROOT / "reports"
FIGURE_DIR = OUTPUT_ROOT / "figures"

RUN_TIMESTAMP = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")

RUN_ROOT = OUTPUT_ROOT / "statistical_significance_block_bootstrap" / f"run_{RUN_ID}"

TABLE_RUN_DIR = RUN_ROOT / "tables"
RETURN_DIR = RUN_ROOT / "returns"
REPORT_RUN_DIR = RUN_ROOT / "reports"
FIGURE_RUN_DIR = RUN_ROOT / "figures"
PAPER_FIGURE_DIR = RUN_ROOT / "paper_figures"

for d in [
    OUTPUT_ROOT,
    TABLE_DIR,
    REPORT_DIR,
    FIGURE_DIR,
    RUN_ROOT,
    TABLE_RUN_DIR,
    RETURN_DIR,
    REPORT_RUN_DIR,
    FIGURE_RUN_DIR,
    PAPER_FIGURE_DIR,
]:
    d.mkdir(parents=True, exist_ok=True)

print("=" * 80)
print("ROMA-TWETF R3: Statistical Significance and Block Bootstrap")
print("=" * 80)
print("Timestamp UTC       :", RUN_TIMESTAMP)
print("Run ID              :", RUN_ID)
print("ROMA R2 run ID      :", ROMA_R2_RUN_ID)
print("ROMA R2 root        :", ROMA_R2_ROOT)
print("R3 input index      :", ROMA_R3_INPUT_INDEX_PATH)
print("AURORA N10 root     :", AURORA_NOTEBOOK10_ROOT)
print("Run root            :", RUN_ROOT)
print("=" * 80)

if not ROMA_R3_INPUT_INDEX_PATH.exists():
    raise FileNotFoundError(f"Missing R3 input index: {ROMA_R3_INPUT_INDEX_PATH}")

# ============================================================
# 2. Global settings
# ============================================================

ANNUALIZATION = 252
RANDOM_STATE = 42

PRIMARY_SPLIT = "strict_test_only"
SECONDARY_SPLIT = "oos_validation_plus_test"

N_BOOTSTRAP = 5000
PRIMARY_BLOCK_LENGTH = 20
BLOCK_LENGTHS = [5, 10, 20, 40]

PRIMARY_ROMA_POLICY = "ROMA_P4_balanced_regime_blend"

ROMA_POLICIES = [
    "ROMA_P0_validation_selected_20_60_blend",
    "ROMA_P1_more20_return_seeking",
    "ROMA_P2_20d_only_return_seeking",
    "ROMA_P3_conservative_blend",
    "ROMA_P4_balanced_regime_blend",
]

PRIMARY_BENCHMARKS = [
    "B12_ma_timing_equal_weight",
    "B6_00881_only",
    "B3_0050_only",
    "B1_equal_weight_all_etfs",
    "B10_momentum_top2_63d",
    "B15_minimum_variance_126d",
]

AURORA_COMPARISON_POLICIES = [
    "AURORA10_UAMV_B_more60_defensive",
    "AURORA10_UAMV_D_low_turnover",
    "AURORA10_UAMV_E_no_regime_tilt_control",
    "AURORA10_UAMV_A_balanced",
    "AURORA10_UAMV_C_more60_growth",
    "AURORA10_validation_selected_UAMV",
]

BOOTSTRAP_METRICS = [
    "diff_total_return",
    "diff_annual_return",
    "diff_annual_volatility",
    "diff_sharpe",
    "diff_sortino",
    "drawdown_improvement",
    "diff_calmar",
    "annualized_mean_excess_return",
    "excess_return_hit_rate",
]

# ============================================================
# 3. Utility functions
# ============================================================

def save_json(path, obj):
    Path(path).write_text(
        json.dumps(obj, indent=2, ensure_ascii=False, default=str),
        encoding="utf-8",
    )

def sha256_file(path, chunk_size=1024 * 1024):
    path = Path(path)
    h = hashlib.sha256()

    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(chunk_size), b""):
            h.update(chunk)

    return h.hexdigest()

def make_file_manifest(root):
    root = Path(root)
    rows = []

    for p in sorted(root.rglob("*")):
        if p.is_file():
            stat = p.stat()
            rows.append({
                "path": p.relative_to(root).as_posix(),
                "size_bytes": int(stat.st_size),
                "modified_utc": datetime.fromtimestamp(
                    stat.st_mtime,
                    timezone.utc,
                ).strftime("%Y-%m-%dT%H:%M:%SZ"),
                "sha256": sha256_file(p),
            })

    return pd.DataFrame(rows)

def safe_name(x):
    return (
        str(x)
        .replace("/", "_")
        .replace("\\", "_")
        .replace(":", "_")
        .replace(" ", "_")
        .replace(".", "_")
        .replace("%", "pct")
        .replace("-", "_")
        .replace("+", "plus")
    )

def read_table(path):
    path = Path(path)

    if not path.exists():
        raise FileNotFoundError(path)

    if path.suffix.lower() == ".parquet":
        return pd.read_parquet(path)

    if path.suffix.lower() == ".csv":
        return pd.read_csv(path)

    raise ValueError(f"Unsupported file type: {path}")

def find_existing_path(candidates):
    for p in candidates:
        p = Path(p)
        if p.exists():
            return p
    return None

def standardize_returns_table(df, table_name="returns_table"):
    """
    Standardize returns table so it always has:
      date, policy_name, split, net_return

    Handles date stored as:
      - date column
      - DatetimeIndex
      - index-like column
    """
    out = df.copy()

    if "date" in out.columns:
        out["date"] = pd.to_datetime(out["date"])
    else:
        idx = out.index

        if isinstance(idx, pd.DatetimeIndex):
            out = out.reset_index()
            first_col = out.columns[0]
            out = out.rename(columns={first_col: "date"})
            out["date"] = pd.to_datetime(out["date"])
        else:
            possible_date_cols = [
                c for c in out.columns
                if str(c).lower() in [
                    "index",
                    "__index_level_0__",
                    "datetime",
                    "timestamp",
                    "time",
                ]
            ]

            recovered = False

            for c in possible_date_cols:
                try:
                    parsed = pd.to_datetime(out[c])
                    if parsed.notna().mean() > 0.95:
                        out = out.rename(columns={c: "date"})
                        out["date"] = parsed
                        recovered = True
                        break
                except Exception:
                    pass

            if not recovered:
                raise KeyError(
                    f"{table_name} has no 'date' column and date could not be recovered. "
                    f"Columns: {list(out.columns)}"
                )

    if "net_return" not in out.columns:
        candidate_return_cols = [
            "return",
            "returns",
            "daily_return",
            "strategy_return",
            "portfolio_return",
            "ret",
        ]

        found = None
        for c in candidate_return_cols:
            if c in out.columns:
                found = c
                break

        if found is not None:
            out = out.rename(columns={found: "net_return"})
        else:
            raise KeyError(
                f"{table_name} has no 'net_return' column. "
                f"Columns: {list(out.columns)}"
            )

    if "policy_name" not in out.columns:
        candidate_policy_cols = [
            "policy",
            "strategy",
            "strategy_name",
            "model_name",
            "portfolio",
        ]

        found = None
        for c in candidate_policy_cols:
            if c in out.columns:
                found = c
                break

        if found is not None:
            out = out.rename(columns={found: "policy_name"})
        else:
            raise KeyError(
                f"{table_name} has no 'policy_name' column. "
                f"Columns: {list(out.columns)}"
            )

    if "split" not in out.columns:
        out["split"] = "all"

    out["date"] = pd.to_datetime(out["date"])
    out["policy_name"] = out["policy_name"].astype(str)
    out["split"] = out["split"].astype(str)
    out["net_return"] = pd.to_numeric(out["net_return"], errors="coerce")

    out = out.dropna(subset=["date", "policy_name", "net_return"])
    out = out.sort_values(["policy_name", "date"]).reset_index(drop=True)

    return out

def calculate_drawdown(equity):
    equity = pd.Series(equity).astype(float)
    running_max = equity.cummax()
    return equity / running_max - 1.0

def performance_metrics(r, annualization=ANNUALIZATION):
    r = pd.Series(r).dropna().astype(float)

    if len(r) == 0:
        return {
            "n_days": 0,
            "total_return": np.nan,
            "annual_return": np.nan,
            "annual_volatility": np.nan,
            "sharpe": np.nan,
            "sortino": np.nan,
            "max_drawdown": np.nan,
            "calmar": np.nan,
            "final_equity": np.nan,
            "mean_daily_return": np.nan,
            "daily_volatility": np.nan,
            "positive_day_rate": np.nan,
        }

    equity = (1.0 + r).cumprod()
    total_return = float(equity.iloc[-1] - 1.0)

    n = len(r)
    annual_return = float(equity.iloc[-1] ** (annualization / n) - 1.0)

    daily_vol = float(r.std(ddof=1)) if n > 1 else np.nan
    annual_vol = float(daily_vol * np.sqrt(annualization)) if np.isfinite(daily_vol) else np.nan

    mean_daily = float(r.mean())
    sharpe = float((mean_daily / daily_vol) * np.sqrt(annualization)) if daily_vol and daily_vol > 0 else np.nan

    downside = r[r < 0]
    downside_vol = float(downside.std(ddof=1)) if len(downside) > 1 else np.nan
    sortino = float((mean_daily / downside_vol) * np.sqrt(annualization)) if np.isfinite(downside_vol) and downside_vol > 0 else np.nan

    drawdown = calculate_drawdown(equity)
    max_drawdown = float(drawdown.min())

    calmar = float(annual_return / abs(max_drawdown)) if max_drawdown < 0 else np.nan

    return {
        "n_days": int(n),
        "total_return": total_return,
        "annual_return": annual_return,
        "annual_volatility": annual_vol,
        "sharpe": sharpe,
        "sortino": sortino,
        "max_drawdown": max_drawdown,
        "calmar": calmar,
        "final_equity": float(equity.iloc[-1]),
        "mean_daily_return": mean_daily,
        "daily_volatility": daily_vol,
        "positive_day_rate": float((r > 0).mean()),
    }

def paired_difference_metrics(strategy_returns, benchmark_returns):
    s = pd.Series(strategy_returns).astype(float)
    b = pd.Series(benchmark_returns).astype(float)

    common = s.index.intersection(b.index).sort_values()
    s = s.loc[common]
    b = b.loc[common]

    ps = performance_metrics(s)
    pb = performance_metrics(b)

    excess = s - b

    return {
        "n_days": int(len(common)),
        "strategy_total_return": ps["total_return"],
        "benchmark_total_return": pb["total_return"],
        "diff_total_return": ps["total_return"] - pb["total_return"],
        "strategy_annual_return": ps["annual_return"],
        "benchmark_annual_return": pb["annual_return"],
        "diff_annual_return": ps["annual_return"] - pb["annual_return"],
        "strategy_annual_volatility": ps["annual_volatility"],
        "benchmark_annual_volatility": pb["annual_volatility"],
        "diff_annual_volatility": ps["annual_volatility"] - pb["annual_volatility"],
        "strategy_sharpe": ps["sharpe"],
        "benchmark_sharpe": pb["sharpe"],
        "diff_sharpe": ps["sharpe"] - pb["sharpe"],
        "strategy_sortino": ps["sortino"],
        "benchmark_sortino": pb["sortino"],
        "diff_sortino": ps["sortino"] - pb["sortino"],
        "strategy_max_drawdown": ps["max_drawdown"],
        "benchmark_max_drawdown": pb["max_drawdown"],
        "drawdown_improvement": ps["max_drawdown"] - pb["max_drawdown"],
        "strategy_calmar": ps["calmar"],
        "benchmark_calmar": pb["calmar"],
        "diff_calmar": ps["calmar"] - pb["calmar"],
        "annualized_mean_excess_return": float(excess.mean() * ANNUALIZATION),
        "excess_return_hit_rate": float((excess > 0).mean()),
        "mean_daily_excess_return": float(excess.mean()),
        "daily_tracking_error": float(excess.std(ddof=1)),
    }

def circular_block_indices(n, block_length, rng):
    if n <= 0:
        return np.array([], dtype=int)

    idx = []

    while len(idx) < n:
        start = rng.integers(0, n)
        block = [(start + k) % n for k in range(block_length)]
        idx.extend(block)

    return np.asarray(idx[:n], dtype=int)

def bootstrap_paired_metrics(
    strategy_returns,
    benchmark_returns,
    n_bootstrap=N_BOOTSTRAP,
    block_length=PRIMARY_BLOCK_LENGTH,
    random_state=RANDOM_STATE,
):
    s = pd.Series(strategy_returns).dropna().astype(float)
    b = pd.Series(benchmark_returns).dropna().astype(float)

    common = s.index.intersection(b.index).sort_values()
    s = s.loc[common]
    b = b.loc[common]

    n = len(common)

    if n == 0:
        return pd.DataFrame()

    s_values = s.values
    b_values = b.values

    rng = np.random.default_rng(random_state)

    rows = []

    for i in range(n_bootstrap):
        boot_idx = circular_block_indices(n, block_length, rng)

        s_boot = pd.Series(s_values[boot_idx])
        b_boot = pd.Series(b_values[boot_idx])

        m = paired_difference_metrics(s_boot, b_boot)
        m["bootstrap_id"] = i
        m["block_length"] = block_length
        rows.append(m)

    return pd.DataFrame(rows)

def iid_bootstrap_paired_metrics(
    strategy_returns,
    benchmark_returns,
    n_bootstrap=N_BOOTSTRAP,
    random_state=RANDOM_STATE,
):
    s = pd.Series(strategy_returns).dropna().astype(float)
    b = pd.Series(benchmark_returns).dropna().astype(float)

    common = s.index.intersection(b.index).sort_values()
    s = s.loc[common]
    b = b.loc[common]

    n = len(common)

    if n == 0:
        return pd.DataFrame()

    s_values = s.values
    b_values = b.values

    rng = np.random.default_rng(random_state)

    rows = []

    for i in range(n_bootstrap):
        boot_idx = rng.integers(0, n, size=n)

        s_boot = pd.Series(s_values[boot_idx])
        b_boot = pd.Series(b_values[boot_idx])

        m = paired_difference_metrics(s_boot, b_boot)
        m["bootstrap_id"] = i
        m["block_length"] = 1
        rows.append(m)

    return pd.DataFrame(rows)

def summarize_bootstrap(boot_df, observed, metrics=BOOTSTRAP_METRICS):
    rows = []

    for metric in metrics:
        if metric not in boot_df.columns:
            continue

        vals = boot_df[metric].dropna().astype(float)

        if len(vals) == 0:
            continue

        obs = float(observed.get(metric, np.nan))

        ci_2p5 = float(vals.quantile(0.025))
        ci_50 = float(vals.quantile(0.500))
        ci_97p5 = float(vals.quantile(0.975))

        rows.append({
            "metric": metric,
            "observed": obs,
            "bootstrap_mean": float(vals.mean()),
            "bootstrap_std": float(vals.std(ddof=1)),
            "ci_2p5": ci_2p5,
            "ci_50": ci_50,
            "ci_97p5": ci_97p5,
            "prob_positive": float((vals > 0).mean()),
            "prob_negative": float((vals < 0).mean()),
            "significant_positive_95": bool(ci_2p5 > 0),
            "significant_negative_95": bool(ci_97p5 < 0),
            "not_significant_95": bool((ci_2p5 <= 0) and (ci_97p5 >= 0)),
        })

    return pd.DataFrame(rows)

def newey_west_se(x, lag=None):
    x = pd.Series(x).dropna().astype(float).values
    n = len(x)

    if n < 2:
        return np.nan

    x_centered = x - x.mean()

    if lag is None:
        lag = int(np.floor(4 * (n / 100) ** (2 / 9)))
        lag = max(1, lag)

    gamma0 = np.dot(x_centered, x_centered) / n
    var = gamma0

    for l in range(1, lag + 1):
        weight = 1.0 - l / (lag + 1.0)
        gamma = np.dot(x_centered[l:], x_centered[:-l]) / n
        var += 2.0 * weight * gamma

    se = np.sqrt(var / n)
    return float(se)

def hac_mean_test(excess_returns, annualization=ANNUALIZATION):
    x = pd.Series(excess_returns).dropna().astype(float)
    n = len(x)

    if n < 3:
        return {
            "n_days": n,
            "mean_daily_excess_return": np.nan,
            "annualized_mean_excess_return": np.nan,
            "newey_west_se_daily": np.nan,
            "t_stat": np.nan,
            "p_value_two_sided": np.nan,
            "lag": np.nan,
        }

    lag = int(np.floor(4 * (n / 100) ** (2 / 9)))
    lag = max(1, lag)

    mean_x = float(x.mean())
    se = newey_west_se(x, lag=lag)
    t_stat = mean_x / se if se and se > 0 else np.nan

    if HAS_SCIPY and np.isfinite(t_stat):
        p_val = float(2.0 * (1.0 - stats.norm.cdf(abs(t_stat))))
    else:
        p_val = np.nan

    return {
        "n_days": int(n),
        "mean_daily_excess_return": mean_x,
        "annualized_mean_excess_return": float(mean_x * annualization),
        "newey_west_se_daily": se,
        "t_stat": float(t_stat) if np.isfinite(t_stat) else np.nan,
        "p_value_two_sided": p_val,
        "lag": int(lag),
    }

def save_table(df, local_name, global_name):
    local_path = TABLE_RUN_DIR / local_name
    global_path = TABLE_DIR / global_name

    df.to_csv(local_path, index=False)
    df.to_csv(global_path, index=False)

    return local_path, global_path

def returns_to_matrix(returns_df, split_name=None, policy_filter=None):
    df = standardize_returns_table(returns_df, table_name="returns_to_matrix_input")

    if split_name is not None and "split" in df.columns:
        split_values = sorted(df["split"].dropna().astype(str).unique().tolist())

        if split_name in split_values:
            df = df[df["split"].astype(str) == split_name].copy()
        elif split_name == PRIMARY_SPLIT:
            alternatives = [
                "aligned_strict_test",
                "strict_test",
                "test",
                "test_only",
                "fold_test_out_of_sample",
            ]

            found_alt = None
            for alt in alternatives:
                if alt in split_values:
                    found_alt = alt
                    break

            if found_alt is not None:
                print(f"Using split '{found_alt}' as alternative for '{split_name}'.")
                df = df[df["split"].astype(str) == found_alt].copy()
            else:
                print(
                    f"Split '{split_name}' not found. Available splits: {split_values}. "
                    "Using all rows and relying on date alignment."
                )

        elif split_name == SECONDARY_SPLIT:
            alternatives = [
                "oos",
                "validation_plus_test",
                "oos_validation_plus_test",
                "all",
            ]

            found_alt = None
            for alt in alternatives:
                if alt in split_values:
                    found_alt = alt
                    break

            if found_alt is not None:
                print(f"Using split '{found_alt}' as alternative for '{split_name}'.")
                df = df[df["split"].astype(str) == found_alt].copy()
            else:
                print(
                    f"Split '{split_name}' not found. Available splits: {split_values}. "
                    "Using all rows and relying on date alignment."
                )

    if policy_filter is not None:
        df = df[df["policy_name"].isin(policy_filter)].copy()

    mat = (
        df
        .pivot_table(index="date", columns="policy_name", values="net_return", aggfunc="last")
        .sort_index()
    )

    return mat

# ============================================================
# 4. Load ROMA R2 outputs
# ============================================================

print("\n" + "=" * 80)
print("Step 1: Loading ROMA R2 outputs")
print("=" * 80)

r3_input = pd.read_csv(ROMA_R3_INPUT_INDEX_PATH)

if r3_input.empty:
    raise ValueError("R3 input index is empty.")

returns_path = Path(r3_input.loc[0, "returns_path_parquet"])
if not returns_path.exists():
    returns_path = Path(r3_input.loc[0, "returns_path_csv"])

weights_path = Path(r3_input.loc[0, "weights_path_parquet"])
if not weights_path.exists():
    weights_path = Path(r3_input.loc[0, "weights_path_csv"])

performance_table_path = Path(r3_input.loc[0, "performance_table"])
ranking_table_path = Path(r3_input.loc[0, "ranking_table"])

roma_returns_raw = read_table(returns_path)
roma_weights = read_table(weights_path)
roma_performance = read_table(performance_table_path)
roma_rankings = read_table(ranking_table_path)

roma_returns = standardize_returns_table(roma_returns_raw, table_name="ROMA R2 returns")

for df, date_cols in [
    (roma_weights, ["date"]),
    (roma_performance, ["start_date", "end_date"]),
    (roma_rankings, ["start_date", "end_date"]),
]:
    for c in date_cols:
        if c in df.columns:
            df[c] = pd.to_datetime(df[c])

print("ROMA returns shape    :", roma_returns.shape)
print("ROMA weights shape    :", roma_weights.shape)
print("ROMA performance shape:", roma_performance.shape)
print("ROMA rankings shape   :", roma_rankings.shape)
print("ROMA returns date     :", roma_returns["date"].min(), "to", roma_returns["date"].max())
print("ROMA policies         :", sorted(roma_returns["policy_name"].unique().tolist()))

# ============================================================
# 5. Load AURORA Notebook 10 returns
# ============================================================

print("\n" + "=" * 80)
print("Step 2: Loading AURORA Notebook 10 returns if available")
print("=" * 80)

aurora_returns_path = find_existing_path(AURORA_RETURNS_CANDIDATES)
aurora_returns = None

if aurora_returns_path is not None:
    raw_aurora_returns = read_table(aurora_returns_path)

    print("AURORA raw returns path :", aurora_returns_path)
    print("AURORA raw shape        :", raw_aurora_returns.shape)
    print("AURORA raw columns      :", list(raw_aurora_returns.columns))
    print("AURORA raw index type   :", type(raw_aurora_returns.index))

    aurora_returns = standardize_returns_table(
        raw_aurora_returns,
        table_name="AURORA Notebook 10 returns",
    )

    print("AURORA standardized shape:", aurora_returns.shape)
    print("AURORA date range        :", aurora_returns["date"].min(), "to", aurora_returns["date"].max())
    print("AURORA policies          :", sorted(aurora_returns["policy_name"].unique().tolist()))
    print("AURORA split values      :", sorted(aurora_returns["split"].dropna().unique().tolist()))
else:
    print("AURORA Notebook 10 returns not found. ROMA-vs-AURORA tests will be skipped.")

# ============================================================
# 6. Build return matrices
# ============================================================

print("\n" + "=" * 80)
print("Step 3: Building return matrices")
print("=" * 80)

roma_mat_oos = returns_to_matrix(roma_returns, SECONDARY_SPLIT)
roma_mat_test = returns_to_matrix(roma_returns, PRIMARY_SPLIT)

print("ROMA OOS matrix :", roma_mat_oos.shape, roma_mat_oos.index.min(), "to", roma_mat_oos.index.max())
print("ROMA test matrix:", roma_mat_test.shape, roma_mat_test.index.min(), "to", roma_mat_test.index.max())

aurora_mat_test = None
aurora_mat_oos = None

if aurora_returns is not None:
    split_values = sorted(aurora_returns["split"].dropna().astype(str).unique().tolist())
    print("AURORA split values:", split_values)

    aurora_mat_test = returns_to_matrix(
        aurora_returns,
        split_name=PRIMARY_SPLIT,
    )

    print(
        "AURORA test matrix:",
        aurora_mat_test.shape,
        aurora_mat_test.index.min(),
        "to",
        aurora_mat_test.index.max(),
    )

    aurora_mat_oos = returns_to_matrix(
        aurora_returns,
        split_name=SECONDARY_SPLIT,
    )

    print(
        "AURORA OOS/all matrix:",
        aurora_mat_oos.shape,
        aurora_mat_oos.index.min(),
        "to",
        aurora_mat_oos.index.max(),
    )

# ============================================================
# 7. Observed performance tables
# ============================================================

print("\n" + "=" * 80)
print("Step 4: Observed performance tables")
print("=" * 80)

def performance_table_from_matrix(mat, split_name, policy_group_map=None):
    rows = []

    for policy in mat.columns:
        r = mat[policy].dropna()
        m = performance_metrics(r)
        m["policy_name"] = policy
        m["split"] = split_name
        m["policy_group"] = policy_group_map.get(policy, "unknown") if policy_group_map else "unknown"
        rows.append(m)

    out = pd.DataFrame(rows)

    if out.empty:
        return out

    out["rank_total_return"] = out["total_return"].rank(ascending=False, method="min")
    out["rank_annual_return"] = out["annual_return"].rank(ascending=False, method="min")
    out["rank_sharpe"] = out["sharpe"].rank(ascending=False, method="min")
    out["rank_sortino"] = out["sortino"].rank(ascending=False, method="min")
    out["rank_max_drawdown"] = out["max_drawdown"].rank(ascending=False, method="min")
    out["rank_calmar"] = out["calmar"].rank(ascending=False, method="min")

    out["composite_rank"] = (
        out["rank_total_return"]
        + out["rank_annual_return"]
        + out["rank_sharpe"]
        + out["rank_sortino"]
        + out["rank_max_drawdown"]
        + out["rank_calmar"]
    ) / 6.0

    out = out.sort_values(
        ["composite_rank", "sharpe", "total_return"],
        ascending=[True, False, False],
    )

    return out

policy_group_map = {}

for p in roma_mat_test.columns:
    if p in ROMA_POLICIES:
        policy_group_map[p] = "ROMA_return_seeking"
    else:
        policy_group_map[p] = "benchmark"

roma_oos_perf = performance_table_from_matrix(roma_mat_oos, SECONDARY_SPLIT, policy_group_map)
roma_test_perf = performance_table_from_matrix(roma_mat_test, PRIMARY_SPLIT, policy_group_map)

save_table(
    roma_oos_perf,
    local_name="roma_R3_observed_performance_oos.csv",
    global_name=f"table_R3_01_roma_observed_performance_oos_{RUN_ID}.csv",
)

save_table(
    roma_test_perf,
    local_name="roma_R3_observed_performance_strict_test.csv",
    global_name=f"table_R3_02_roma_observed_performance_strict_test_{RUN_ID}.csv",
)

print("Observed strict-test performance:")
print(
    roma_test_perf[
        [
            "policy_name",
            "policy_group",
            "n_days",
            "total_return",
            "annual_return",
            "annual_volatility",
            "sharpe",
            "sortino",
            "max_drawdown",
            "calmar",
            "composite_rank",
        ]
    ].to_string(index=False)
)

# ============================================================
# 8. Pairwise comparison set
# ============================================================

print("\n" + "=" * 80)
print("Step 5: Defining pairwise comparisons")
print("=" * 80)

pair_rows = []

for roma_policy in ROMA_POLICIES:
    if roma_policy not in roma_mat_test.columns:
        continue

    for benchmark in PRIMARY_BENCHMARKS:
        if benchmark not in roma_mat_test.columns:
            continue

        pair_rows.append({
            "comparison_group": "ROMA_vs_benchmark",
            "strategy_policy": roma_policy,
            "benchmark_policy": benchmark,
            "return_source": "ROMA_R2",
            "split": PRIMARY_SPLIT,
        })

available_benchmarks = [
    p for p in roma_mat_test.columns
    if p not in ROMA_POLICIES
]

for benchmark in available_benchmarks:
    pair_rows.append({
        "comparison_group": "best_ROMA_vs_all_benchmarks",
        "strategy_policy": PRIMARY_ROMA_POLICY,
        "benchmark_policy": benchmark,
        "return_source": "ROMA_R2",
        "split": PRIMARY_SPLIT,
    })

if aurora_mat_test is not None:
    available_aurora_policies = set(aurora_mat_test.columns)

    for roma_policy in [PRIMARY_ROMA_POLICY, "ROMA_P2_20d_only_return_seeking"]:
        if roma_policy not in roma_mat_test.columns:
            continue

        for aurora_policy in AURORA_COMPARISON_POLICIES:
            if aurora_policy in available_aurora_policies:
                pair_rows.append({
                    "comparison_group": "ROMA_vs_AURORA",
                    "strategy_policy": roma_policy,
                    "benchmark_policy": aurora_policy,
                    "return_source": "ROMA_R2_and_AURORA_N10",
                    "split": PRIMARY_SPLIT,
                })

pair_index_df = pd.DataFrame(pair_rows)

save_table(
    pair_index_df,
    local_name="roma_R3_pairwise_comparison_index.csv",
    global_name=f"table_R3_03_roma_pairwise_comparison_index_{RUN_ID}.csv",
)

print("Pairwise comparisons:", pair_index_df.shape)
print(pair_index_df.to_string(index=False))

# ============================================================
# 9. Observed pairwise differences and HAC tests
# ============================================================

print("\n" + "=" * 80)
print("Step 6: Observed pairwise differences and HAC tests")
print("=" * 80)

observed_rows = []
hac_rows = []

def get_policy_returns(policy, source):
    if source == "ROMA_R2":
        if policy not in roma_mat_test.columns:
            raise KeyError(policy)
        return roma_mat_test[policy].dropna()

    if source == "AURORA_N10":
        if aurora_mat_test is None or policy not in aurora_mat_test.columns:
            raise KeyError(policy)
        return aurora_mat_test[policy].dropna()

    raise ValueError(source)

for _, row in pair_index_df.iterrows():
    comparison_group = row["comparison_group"]
    strategy_policy = row["strategy_policy"]
    benchmark_policy = row["benchmark_policy"]
    return_source = row["return_source"]

    if return_source == "ROMA_R2":
        s = get_policy_returns(strategy_policy, "ROMA_R2")
        b = get_policy_returns(benchmark_policy, "ROMA_R2")
    elif return_source == "ROMA_R2_and_AURORA_N10":
        s = get_policy_returns(strategy_policy, "ROMA_R2")
        b = get_policy_returns(benchmark_policy, "AURORA_N10")
    else:
        raise ValueError(return_source)

    common = s.index.intersection(b.index).sort_values()
    s = s.loc[common]
    b = b.loc[common]

    obs = paired_difference_metrics(s, b)

    observed_rows.append({
        "comparison_group": comparison_group,
        "strategy_policy": strategy_policy,
        "benchmark_policy": benchmark_policy,
        "return_source": return_source,
        "split": PRIMARY_SPLIT,
        **obs,
    })

    excess = s - b
    hac = hac_mean_test(excess)

    hac_rows.append({
        "comparison_group": comparison_group,
        "strategy_policy": strategy_policy,
        "benchmark_policy": benchmark_policy,
        "return_source": return_source,
        "split": PRIMARY_SPLIT,
        **hac,
    })

observed_pairwise_df = pd.DataFrame(observed_rows)
hac_df = pd.DataFrame(hac_rows)

save_table(
    observed_pairwise_df,
    local_name="roma_R3_observed_pairwise_differences.csv",
    global_name=f"table_R3_04_roma_observed_pairwise_differences_{RUN_ID}.csv",
)

save_table(
    hac_df,
    local_name="roma_R3_hac_newey_west_tests.csv",
    global_name=f"table_R3_05_roma_hac_newey_west_tests_{RUN_ID}.csv",
)

print("Observed pairwise differences:")
print(
    observed_pairwise_df[
        [
            "comparison_group",
            "strategy_policy",
            "benchmark_policy",
            "n_days",
            "diff_total_return",
            "diff_annual_return",
            "diff_sharpe",
            "diff_sortino",
            "drawdown_improvement",
            "diff_calmar",
            "annualized_mean_excess_return",
            "excess_return_hit_rate",
        ]
    ].to_string(index=False)
)

# ============================================================
# 10. Primary circular block bootstrap
# ============================================================

print("\n" + "=" * 80)
print("Step 7: Primary circular block bootstrap")
print("=" * 80)

primary_bootstrap_summary_rows = []
primary_bootstrap_detail_paths = []

primary_pairs = pair_index_df[
    pair_index_df["comparison_group"].isin([
        "ROMA_vs_benchmark",
        "ROMA_vs_AURORA",
    ])
].copy()

for i, row in primary_pairs.iterrows():
    comparison_group = row["comparison_group"]
    strategy_policy = row["strategy_policy"]
    benchmark_policy = row["benchmark_policy"]
    return_source = row["return_source"]

    print(
        f"Bootstrap {i + 1}/{len(primary_pairs)}:",
        comparison_group,
        "|",
        strategy_policy,
        "vs",
        benchmark_policy,
    )

    if return_source == "ROMA_R2":
        s = get_policy_returns(strategy_policy, "ROMA_R2")
        b = get_policy_returns(benchmark_policy, "ROMA_R2")
    elif return_source == "ROMA_R2_and_AURORA_N10":
        s = get_policy_returns(strategy_policy, "ROMA_R2")
        b = get_policy_returns(benchmark_policy, "AURORA_N10")
    else:
        raise ValueError(return_source)

    common = s.index.intersection(b.index).sort_values()
    s = s.loc[common]
    b = b.loc[common]

    observed = paired_difference_metrics(s, b)

    boot_df = bootstrap_paired_metrics(
        s,
        b,
        n_bootstrap=N_BOOTSTRAP,
        block_length=PRIMARY_BLOCK_LENGTH,
        random_state=RANDOM_STATE + i,
    )

    boot_summary = summarize_bootstrap(boot_df, observed)

    boot_summary.insert(0, "comparison_group", comparison_group)
    boot_summary.insert(1, "strategy_policy", strategy_policy)
    boot_summary.insert(2, "benchmark_policy", benchmark_policy)
    boot_summary.insert(3, "return_source", return_source)
    boot_summary.insert(4, "split", PRIMARY_SPLIT)
    boot_summary.insert(5, "block_length", PRIMARY_BLOCK_LENGTH)
    boot_summary.insert(6, "n_bootstrap", N_BOOTSTRAP)

    primary_bootstrap_summary_rows.append(boot_summary)

    if strategy_policy == PRIMARY_ROMA_POLICY or comparison_group == "ROMA_vs_AURORA":
        detail_name = (
            f"bootstrap_detail_{safe_name(comparison_group)}_"
            f"{safe_name(strategy_policy)}_vs_{safe_name(benchmark_policy)}_"
            f"block{PRIMARY_BLOCK_LENGTH}.parquet"
        )
        detail_path = RETURN_DIR / detail_name
        boot_df.to_parquet(detail_path, index=False)

        primary_bootstrap_detail_paths.append({
            "comparison_group": comparison_group,
            "strategy_policy": strategy_policy,
            "benchmark_policy": benchmark_policy,
            "block_length": PRIMARY_BLOCK_LENGTH,
            "n_bootstrap": N_BOOTSTRAP,
            "bootstrap_detail_path": str(detail_path),
        })

primary_bootstrap_summary_df = pd.concat(primary_bootstrap_summary_rows, axis=0, ignore_index=True)
bootstrap_detail_index_df = pd.DataFrame(primary_bootstrap_detail_paths)

save_table(
    primary_bootstrap_summary_df,
    local_name="roma_R3_primary_block_bootstrap_summary.csv",
    global_name=f"table_R3_06_roma_primary_block_bootstrap_summary_{RUN_ID}.csv",
)

save_table(
    bootstrap_detail_index_df,
    local_name="roma_R3_bootstrap_detail_index.csv",
    global_name=f"table_R3_07_roma_bootstrap_detail_index_{RUN_ID}.csv",
)

print("Primary bootstrap summary:")
print(
    primary_bootstrap_summary_df[
        [
            "comparison_group",
            "strategy_policy",
            "benchmark_policy",
            "metric",
            "observed",
            "ci_2p5",
            "ci_97p5",
            "prob_positive",
            "significant_positive_95",
            "significant_negative_95",
            "not_significant_95",
        ]
    ].to_string(index=False)
)

# ============================================================
# 11. Block-length sensitivity for primary ROMA policy
# ============================================================

print("\n" + "=" * 80)
print("Step 8: Block-length sensitivity for primary ROMA policy")
print("=" * 80)

sensitivity_rows = []

sensitivity_pairs = pair_index_df[
    (
        (pair_index_df["strategy_policy"] == PRIMARY_ROMA_POLICY)
        & (pair_index_df["comparison_group"].isin(["ROMA_vs_benchmark", "ROMA_vs_AURORA"]))
    )
].copy()

for _, row in sensitivity_pairs.iterrows():
    comparison_group = row["comparison_group"]
    strategy_policy = row["strategy_policy"]
    benchmark_policy = row["benchmark_policy"]
    return_source = row["return_source"]

    if return_source == "ROMA_R2":
        s = get_policy_returns(strategy_policy, "ROMA_R2")
        b = get_policy_returns(benchmark_policy, "ROMA_R2")
    elif return_source == "ROMA_R2_and_AURORA_N10":
        s = get_policy_returns(strategy_policy, "ROMA_R2")
        b = get_policy_returns(benchmark_policy, "AURORA_N10")
    else:
        continue

    common = s.index.intersection(b.index).sort_values()
    s = s.loc[common]
    b = b.loc[common]

    observed = paired_difference_metrics(s, b)

    for block_length in BLOCK_LENGTHS:
        print(
            "Sensitivity:",
            strategy_policy,
            "vs",
            benchmark_policy,
            "| block",
            block_length,
        )

        boot_df = bootstrap_paired_metrics(
            s,
            b,
            n_bootstrap=N_BOOTSTRAP,
            block_length=block_length,
            random_state=RANDOM_STATE + 1000 + block_length,
        )

        summary = summarize_bootstrap(boot_df, observed)

        summary.insert(0, "comparison_group", comparison_group)
        summary.insert(1, "strategy_policy", strategy_policy)
        summary.insert(2, "benchmark_policy", benchmark_policy)
        summary.insert(3, "return_source", return_source)
        summary.insert(4, "split", PRIMARY_SPLIT)
        summary.insert(5, "block_length", block_length)
        summary.insert(6, "n_bootstrap", N_BOOTSTRAP)

        sensitivity_rows.append(summary)

sensitivity_df = pd.concat(sensitivity_rows, axis=0, ignore_index=True) if sensitivity_rows else pd.DataFrame()

save_table(
    sensitivity_df,
    local_name="roma_R3_block_length_sensitivity.csv",
    global_name=f"table_R3_08_roma_block_length_sensitivity_{RUN_ID}.csv",
)

# ============================================================
# 12. IID bootstrap sensitivity
# ============================================================

print("\n" + "=" * 80)
print("Step 9: IID bootstrap sensitivity for primary ROMA policy")
print("=" * 80)

iid_rows = []

for _, row in sensitivity_pairs.iterrows():
    comparison_group = row["comparison_group"]
    strategy_policy = row["strategy_policy"]
    benchmark_policy = row["benchmark_policy"]
    return_source = row["return_source"]

    if return_source == "ROMA_R2":
        s = get_policy_returns(strategy_policy, "ROMA_R2")
        b = get_policy_returns(benchmark_policy, "ROMA_R2")
    elif return_source == "ROMA_R2_and_AURORA_N10":
        s = get_policy_returns(strategy_policy, "ROMA_R2")
        b = get_policy_returns(benchmark_policy, "AURORA_N10")
    else:
        continue

    common = s.index.intersection(b.index).sort_values()
    s = s.loc[common]
    b = b.loc[common]

    observed = paired_difference_metrics(s, b)

    print("IID sensitivity:", strategy_policy, "vs", benchmark_policy)

    boot_df = iid_bootstrap_paired_metrics(
        s,
        b,
        n_bootstrap=N_BOOTSTRAP,
        random_state=RANDOM_STATE + 2000,
    )

    summary = summarize_bootstrap(boot_df, observed)

    summary.insert(0, "comparison_group", comparison_group)
    summary.insert(1, "strategy_policy", strategy_policy)
    summary.insert(2, "benchmark_policy", benchmark_policy)
    summary.insert(3, "return_source", return_source)
    summary.insert(4, "split", PRIMARY_SPLIT)
    summary.insert(5, "bootstrap_type", "iid")
    summary.insert(6, "n_bootstrap", N_BOOTSTRAP)

    iid_rows.append(summary)

iid_sensitivity_df = pd.concat(iid_rows, axis=0, ignore_index=True) if iid_rows else pd.DataFrame()

save_table(
    iid_sensitivity_df,
    local_name="roma_R3_iid_bootstrap_sensitivity.csv",
    global_name=f"table_R3_09_roma_iid_bootstrap_sensitivity_{RUN_ID}.csv",
)

# ============================================================
# 13. Paper decision table
# ============================================================

print("\n" + "=" * 80)
print("Step 10: Building paper decision table")
print("=" * 80)

def decision_from_summary(row):
    if bool(row["significant_positive_95"]):
        return "significantly_positive_at_95pct_bootstrap"

    if bool(row["significant_negative_95"]):
        return "significantly_negative_at_95pct_bootstrap"

    return "not_significant_at_95pct_bootstrap"

decision_metrics = [
    "diff_total_return",
    "diff_sharpe",
    "diff_sortino",
    "drawdown_improvement",
    "diff_calmar",
    "annualized_mean_excess_return",
]

decision_df = primary_bootstrap_summary_df[
    primary_bootstrap_summary_df["metric"].isin(decision_metrics)
].copy()

decision_df["decision"] = decision_df.apply(decision_from_summary, axis=1)

decision_df["plain_language_interpretation"] = decision_df.apply(
    lambda r: (
        "ROMA is significantly better on this metric."
        if r["decision"] == "significantly_positive_at_95pct_bootstrap"
        else (
            "ROMA is significantly worse on this metric."
            if r["decision"] == "significantly_negative_at_95pct_bootstrap"
            else "No statistically significant ROMA advantage on this metric."
        )
    ),
    axis=1,
)

decision_df["metric_interpretation"] = decision_df["metric"].map({
    "diff_total_return": "Positive means ROMA has higher cumulative return than comparator.",
    "diff_sharpe": "Positive means ROMA has higher Sharpe ratio than comparator.",
    "diff_sortino": "Positive means ROMA has higher Sortino ratio than comparator.",
    "drawdown_improvement": "Positive means ROMA has less severe maximum drawdown than comparator.",
    "diff_calmar": "Positive means ROMA has higher Calmar ratio than comparator.",
    "annualized_mean_excess_return": "Positive means ROMA has higher average daily return annualized.",
})

save_table(
    decision_df,
    local_name="roma_R3_paper_decision_table.csv",
    global_name=f"table_R3_10_roma_paper_decision_table_{RUN_ID}.csv",
)

print("Paper decision table:")
print(
    decision_df[
        [
            "comparison_group",
            "strategy_policy",
            "benchmark_policy",
            "metric",
            "observed",
            "ci_2p5",
            "ci_97p5",
            "decision",
            "plain_language_interpretation",
        ]
    ].to_string(index=False)
)

# ============================================================
# 14. Diagnostic summary
# ============================================================

print("\n" + "=" * 80)
print("Step 11: Diagnostic summary")
print("=" * 80)

diagnostic_rows = []

best_roma = (
    roma_test_perf[roma_test_perf["policy_group"] == "ROMA_return_seeking"]
    .sort_values("composite_rank")
    .iloc[0]
)

best_benchmark = (
    roma_test_perf[roma_test_perf["policy_group"] == "benchmark"]
    .sort_values("composite_rank")
    .iloc[0]
)

best_overall = roma_test_perf.sort_values("composite_rank").iloc[0]

diagnostic_rows.append({
    "section": "strict_test_observed_performance",
    "finding": "best_ROMA_policy",
    "policy": best_roma["policy_name"],
    "comparator": None,
    "n_days": int(best_roma["n_days"]),
    "total_return": best_roma["total_return"],
    "sharpe": best_roma["sharpe"],
    "sortino": best_roma["sortino"],
    "max_drawdown": best_roma["max_drawdown"],
    "calmar": best_roma["calmar"],
    "composite_rank": best_roma["composite_rank"],
    "interpretation": "Best ROMA policy by strict-test composite rank.",
})

diagnostic_rows.append({
    "section": "strict_test_observed_performance",
    "finding": "best_benchmark_policy",
    "policy": best_benchmark["policy_name"],
    "comparator": None,
    "n_days": int(best_benchmark["n_days"]),
    "total_return": best_benchmark["total_return"],
    "sharpe": best_benchmark["sharpe"],
    "sortino": best_benchmark["sortino"],
    "max_drawdown": best_benchmark["max_drawdown"],
    "calmar": best_benchmark["calmar"],
    "composite_rank": best_benchmark["composite_rank"],
    "interpretation": "Best benchmark by strict-test composite rank.",
})

primary_vs_best = decision_df[
    (decision_df["strategy_policy"] == PRIMARY_ROMA_POLICY)
    & (decision_df["benchmark_policy"] == best_benchmark["policy_name"])
].copy()

for metric in decision_metrics:
    rows = primary_vs_best[primary_vs_best["metric"] == metric]

    if rows.empty:
        continue

    r = rows.iloc[0]

    diagnostic_rows.append({
        "section": "bootstrap_decision_primary_ROMA_vs_best_benchmark",
        "finding": metric,
        "policy": PRIMARY_ROMA_POLICY,
        "comparator": best_benchmark["policy_name"],
        "observed_difference": r["observed"],
        "ci_2p5": r["ci_2p5"],
        "ci_97p5": r["ci_97p5"],
        "decision": r["decision"],
        "interpretation": r["plain_language_interpretation"],
    })

if aurora_mat_test is not None:
    roma_aurora_decisions = decision_df[
        (decision_df["comparison_group"] == "ROMA_vs_AURORA")
        & (decision_df["strategy_policy"] == PRIMARY_ROMA_POLICY)
        & (decision_df["benchmark_policy"] == "AURORA10_UAMV_B_more60_defensive")
    ].copy()

    for metric in decision_metrics:
        rows = roma_aurora_decisions[roma_aurora_decisions["metric"] == metric]

        if rows.empty:
            continue

        r = rows.iloc[0]

        diagnostic_rows.append({
            "section": "bootstrap_decision_primary_ROMA_vs_AURORA10_UAMV_B",
            "finding": metric,
            "policy": PRIMARY_ROMA_POLICY,
            "comparator": "AURORA10_UAMV_B_more60_defensive",
            "observed_difference": r["observed"],
            "ci_2p5": r["ci_2p5"],
            "ci_97p5": r["ci_97p5"],
            "decision": r["decision"],
            "interpretation": r["plain_language_interpretation"],
        })

diagnostic_df = pd.DataFrame(diagnostic_rows)

save_table(
    diagnostic_df,
    local_name="roma_R3_diagnostic_summary.csv",
    global_name=f"table_R3_11_roma_diagnostic_summary_{RUN_ID}.csv",
)

print("Diagnostic summary:")
print(diagnostic_df.to_string(index=False))

# ============================================================
# 15. Figures
# ============================================================

print("\n" + "=" * 80)
print("Step 12: Creating figures")
print("=" * 80)

figure_paths = []

def plot_observed_bar(metric, split_name=PRIMARY_SPLIT, top_n=20):
    perf = roma_test_perf.copy() if split_name == PRIMARY_SPLIT else roma_oos_perf.copy()

    if perf.empty or metric not in perf.columns:
        return None

    perf = perf.sort_values(metric, ascending=False).head(top_n)

    colors = [
        "#1f77b4" if g == "ROMA_return_seeking" else "#7f7f7f"
        for g in perf["policy_group"]
    ]

    plt.figure(figsize=(12, 7))
    plt.barh(perf["policy_name"], perf[metric], color=colors)
    plt.gca().invert_yaxis()
    plt.xlabel(metric)
    plt.title(f"ROMA R3 observed {metric} ranking - {split_name}")
    plt.grid(axis="x", alpha=0.3)
    plt.tight_layout()

    out = FIGURE_RUN_DIR / f"roma_R3_observed_{safe_name(metric)}_{safe_name(split_name)}.png"
    plt.savefig(out, dpi=180)
    plt.close()

    return out

def plot_bootstrap_ci_for_primary(metric):
    dfp = primary_bootstrap_summary_df[
        (primary_bootstrap_summary_df["strategy_policy"] == PRIMARY_ROMA_POLICY)
        & (primary_bootstrap_summary_df["metric"] == metric)
        & (primary_bootstrap_summary_df["comparison_group"] == "ROMA_vs_benchmark")
    ].copy()

    if dfp.empty:
        return None

    dfp = dfp.sort_values("observed")

    y = np.arange(len(dfp))
    x = dfp["observed"].values
    xerr_low = x - dfp["ci_2p5"].values
    xerr_high = dfp["ci_97p5"].values - x

    plt.figure(figsize=(11, 6))
    plt.errorbar(
        x,
        y,
        xerr=[xerr_low, xerr_high],
        fmt="o",
        capsize=4,
        color="#1f77b4",
    )
    plt.axvline(0.0, color="black", linestyle="--", linewidth=1)
    plt.yticks(y, dfp["benchmark_policy"])
    plt.xlabel(metric)
    plt.title(f"{PRIMARY_ROMA_POLICY} vs benchmarks: bootstrap 95% CI for {metric}")
    plt.grid(axis="x", alpha=0.3)
    plt.tight_layout()

    out = FIGURE_RUN_DIR / f"roma_R3_primary_bootstrap_ci_{safe_name(metric)}.png"
    plt.savefig(out, dpi=180)
    plt.close()

    return out

def plot_romavsaura_ci(metric):
    dfp = primary_bootstrap_summary_df[
        (primary_bootstrap_summary_df["comparison_group"] == "ROMA_vs_AURORA")
        & (primary_bootstrap_summary_df["metric"] == metric)
    ].copy()

    if dfp.empty:
        return None

    dfp["pair"] = dfp["strategy_policy"] + " vs " + dfp["benchmark_policy"]
    dfp = dfp.sort_values("observed")

    y = np.arange(len(dfp))
    x = dfp["observed"].values
    xerr_low = x - dfp["ci_2p5"].values
    xerr_high = dfp["ci_97p5"].values - x

    plt.figure(figsize=(12, max(5, 0.45 * len(dfp))))
    plt.errorbar(
        x,
        y,
        xerr=[xerr_low, xerr_high],
        fmt="o",
        capsize=4,
        color="#d62728",
    )
    plt.axvline(0.0, color="black", linestyle="--", linewidth=1)
    plt.yticks(y, dfp["pair"])
    plt.xlabel(metric)
    plt.title(f"ROMA vs AURORA: bootstrap 95% CI for {metric}")
    plt.grid(axis="x", alpha=0.3)
    plt.tight_layout()

    out = FIGURE_RUN_DIR / f"roma_R3_romavsaura_bootstrap_ci_{safe_name(metric)}.png"
    plt.savefig(out, dpi=180)
    plt.close()

    return out

for metric in ["total_return", "sharpe", "max_drawdown", "calmar"]:
    p = plot_observed_bar(metric)
    if p is not None:
        figure_paths.append(p)

for metric in ["diff_total_return", "diff_sharpe", "drawdown_improvement", "diff_calmar"]:
    p = plot_bootstrap_ci_for_primary(metric)
    if p is not None:
        figure_paths.append(p)

    p = plot_romavsaura_ci(metric)
    if p is not None:
        figure_paths.append(p)

figure_index_df = pd.DataFrame({
    "figure_path": [str(p) for p in figure_paths],
    "description": [Path(p).stem for p in figure_paths],
})

save_table(
    figure_index_df,
    local_name="roma_R3_figure_index.csv",
    global_name=f"table_R3_12_roma_figure_index_{RUN_ID}.csv",
)

# ============================================================
# 16. Export return matrices for Notebook 13
# ============================================================

print("\n" + "=" * 80)
print("Step 13: Exporting return matrices for Notebook 13")
print("=" * 80)

roma_test_matrix_path = RETURN_DIR / "roma_R3_strict_test_return_matrix.parquet"
roma_oos_matrix_path = RETURN_DIR / "roma_R3_oos_return_matrix.parquet"

roma_mat_test.to_parquet(roma_test_matrix_path)
roma_mat_oos.to_parquet(roma_oos_matrix_path)

if aurora_mat_test is not None:
    aurora_test_matrix_path = RETURN_DIR / "aurora_N10_strict_test_return_matrix_loaded_in_R3.parquet"
    aurora_mat_test.to_parquet(aurora_test_matrix_path)
else:
    aurora_test_matrix_path = None

compact_pairwise_path = RETURN_DIR / "roma_R3_observed_pairwise_differences.parquet"
observed_pairwise_df.to_parquet(compact_pairwise_path, index=False)

# ============================================================
# 17. Notebook 13 input index
# ============================================================

print("\n" + "=" * 80)
print("Step 14: Creating Notebook 13 input index")
print("=" * 80)

notebook13_index = pd.DataFrame([
    {
        "run_id": RUN_ID,
        "project_code": PROJECT_CODE,
        "notebook": "R3_ROMA_statistical_significance_block_bootstrap.ipynb",
        "roma_r2_run_id": ROMA_R2_RUN_ID,
        "roma_r3_input_index": str(ROMA_R3_INPUT_INDEX_PATH),
        "roma_r2_returns_path": str(returns_path),
        "roma_r2_weights_path": str(weights_path),
        "roma_test_return_matrix_path": str(roma_test_matrix_path),
        "roma_oos_return_matrix_path": str(roma_oos_matrix_path),
        "aurora_notebook10_run_id": AURORA_NOTEBOOK10_RUN_ID,
        "aurora_returns_path": str(aurora_returns_path) if aurora_returns_path is not None else None,
        "aurora_test_return_matrix_path": str(aurora_test_matrix_path) if aurora_test_matrix_path is not None else None,
        "observed_pairwise_differences_path": str(compact_pairwise_path),
        "paper_decision_table_path": str(TABLE_RUN_DIR / "roma_R3_paper_decision_table.csv"),
        "primary_bootstrap_summary_path": str(TABLE_RUN_DIR / "roma_R3_primary_block_bootstrap_summary.csv"),
        "diagnostic_summary_path": str(TABLE_RUN_DIR / "roma_R3_diagnostic_summary.csv"),
        "primary_roma_policy": PRIMARY_ROMA_POLICY,
        "primary_split": PRIMARY_SPLIT,
        "n_bootstrap": N_BOOTSTRAP,
        "primary_block_length": PRIMARY_BLOCK_LENGTH,
        "strict_test_start": roma_mat_test.index.min(),
        "strict_test_end": roma_mat_test.index.max(),
        "strict_test_n_days": int(len(roma_mat_test)),
    }
])

notebook13_index_path = RUN_ROOT / "NOTEBOOK13_ROMA_AURORA_INPUT_INDEX.csv"
notebook13_index_global_path = TABLE_DIR / f"table_R3_13_NOTEBOOK13_ROMA_AURORA_INPUT_INDEX_{RUN_ID}.csv"

notebook13_index.to_csv(notebook13_index_path, index=False)
notebook13_index.to_csv(notebook13_index_global_path, index=False)

print("Notebook 13 input index:")
print(notebook13_index.to_string(index=False))

# ============================================================
# 18. Validation report and manifest
# ============================================================

print("\n" + "=" * 80)
print("Step 15: Saving validation report and SHA256 manifest")
print("=" * 80)

validation_report = {
    "project_code": PROJECT_CODE,
    "notebook": "R3_ROMA_statistical_significance_block_bootstrap.ipynb",
    "run_timestamp_utc": RUN_TIMESTAMP,
    "run_id": RUN_ID,
    "roma_r2_run_id": ROMA_R2_RUN_ID,
    "purpose": (
        "Statistical inference for rebuilt ROMA allocation policies under aligned strict-test evaluation. "
        "ROMA is tested as a regime-template baseline and compared with benchmarks and AURORA when available."
    ),
    "primary_split": PRIMARY_SPLIT,
    "secondary_split": SECONDARY_SPLIT,
    "primary_roma_policy": PRIMARY_ROMA_POLICY,
    "primary_benchmarks": PRIMARY_BENCHMARKS,
    "aurora_comparison_policies": AURORA_COMPARISON_POLICIES,
    "n_bootstrap": N_BOOTSTRAP,
    "primary_block_length": PRIMARY_BLOCK_LENGTH,
    "block_length_sensitivity": BLOCK_LENGTHS,
    "input_paths": {
        "roma_r3_input_index": str(ROMA_R3_INPUT_INDEX_PATH),
        "roma_r2_returns": str(returns_path),
        "roma_r2_weights": str(weights_path),
        "aurora_returns": str(aurora_returns_path) if aurora_returns_path is not None else None,
    },
    "output_paths": {
        "run_root": str(RUN_ROOT),
        "tables": str(TABLE_RUN_DIR),
        "returns": str(RETURN_DIR),
        "figures": str(FIGURE_RUN_DIR),
        "reports": str(REPORT_RUN_DIR),
        "notebook13_input_index": str(notebook13_index_path),
    },
    "important_methodological_notes": [
        "ROMA R3 does not perform new model selection.",
        "ROMA policies are evaluated using returns generated by R2 on aligned dates.",
        "Paired circular block bootstrap is used to account for serial dependence.",
        "Positive drawdown_improvement means ROMA had a less severe maximum drawdown than the comparator.",
        "ROMA-vs-AURORA comparisons are included only if AURORA Notebook 10 returns are available.",
        "Notebook 13 will combine ROMA and AURORA evidence into final manuscript tables.",
    ],
    "educational_note": (
        "This notebook is for reproducible financial machine-learning research only. "
        "It does not provide personalized financial advice or performance guarantees."
    ),
}

validation_report_path = REPORT_RUN_DIR / "ROMA_R3_validation_report.json"
validation_report_global_path = REPORT_DIR / f"ROMA_R3_validation_report_{RUN_ID}.json"

save_json(validation_report_path, validation_report)
save_json(validation_report_global_path, validation_report)

manifest_df = make_file_manifest(RUN_ROOT)

manifest_path = REPORT_RUN_DIR / "ROMA_R3_file_manifest_SHA256.csv"
manifest_global_path = REPORT_DIR / f"ROMA_R3_file_manifest_SHA256_{RUN_ID}.csv"

manifest_df.to_csv(manifest_path, index=False)
manifest_df.to_csv(manifest_global_path, index=False)

# ============================================================
# 19. Final summary
# ============================================================

print("\n" + "=" * 80)
print("ROMA-TWETF R3 COMPLETE")
print("=" * 80)
print("Run ID                         :", RUN_ID)
print("Run root                       :", RUN_ROOT)
print("ROMA R2 run ID                 :", ROMA_R2_RUN_ID)
print("ROMA R2 returns                :", returns_path)
print("AURORA returns loaded          :", aurora_returns_path if aurora_returns_path is not None else "None")
print("Observed performance OOS       :", TABLE_DIR / f"table_R3_01_roma_observed_performance_oos_{RUN_ID}.csv")
print("Observed performance test      :", TABLE_DIR / f"table_R3_02_roma_observed_performance_strict_test_{RUN_ID}.csv")
print("Pairwise comparison index      :", TABLE_DIR / f"table_R3_03_roma_pairwise_comparison_index_{RUN_ID}.csv")
print("Observed pairwise differences  :", TABLE_DIR / f"table_R3_04_roma_observed_pairwise_differences_{RUN_ID}.csv")
print("HAC tests                      :", TABLE_DIR / f"table_R3_05_roma_hac_newey_west_tests_{RUN_ID}.csv")
print("Primary bootstrap summary      :", TABLE_DIR / f"table_R3_06_roma_primary_block_bootstrap_summary_{RUN_ID}.csv")
print("Block length sensitivity       :", TABLE_DIR / f"table_R3_08_roma_block_length_sensitivity_{RUN_ID}.csv")
print("IID bootstrap sensitivity      :", TABLE_DIR / f"table_R3_09_roma_iid_bootstrap_sensitivity_{RUN_ID}.csv")
print("Paper decision table           :", TABLE_DIR / f"table_R3_10_roma_paper_decision_table_{RUN_ID}.csv")
print("Diagnostic summary             :", TABLE_DIR / f"table_R3_11_roma_diagnostic_summary_{RUN_ID}.csv")
print("Notebook 13 input index        :", notebook13_index_path)
print("Validation report              :", validation_report_path)
print("Manifest                       :", manifest_path)
print("=" * 80)

print("\nRecommended next notebook:")
print("13_ROMA_AURORA_unified_comparison.ipynb")
print("\nImportant for Notebook 13:")
print("Use this file:")
print(notebook13_index_path)

Mounted at /content/drive
ROMA-TWETF R3: Statistical Significance and Block Bootstrap
Timestamp UTC       : 2026-06-25T03:14:40Z
Run ID              : 20260625_031440
ROMA R2 run ID      : 20260625_025314
ROMA R2 root        : /content/drive/MyDrive/AURORA_TWETF/outputs/ROMA_TWETF/aligned_allocation_backtest/run_20260625_025314
R3 input index      : /content/drive/MyDrive/AURORA_TWETF/outputs/ROMA_TWETF/aligned_allocation_backtest/run_20260625_025314/NOTEBOOK_R3_ROMA_INPUT_INDEX.csv
AURORA N10 root     : /content/drive/MyDrive/AURORA_TWETF/outputs/AURORA_TWETF/uncertainty_aware_mean_variance_allocation/run_20260624_100748
Run root            : /content/drive/MyDrive/AURORA_TWETF/outputs/ROMA_TWETF/statistical_significance_block_bootstrap/run_20260625_031440

Step 1: Loading ROMA R2 outputs
ROMA returns shape    : (16540, 9)
ROMA weights shape    : (16540, 9)
ROMA performance shape: (40, 28)
ROMA rankings shape   : (40, 35)
ROMA returns date     : 2024-02-20 00:00:00 to 2026-03-25 00:00